# Motivação

Do jeito que o agente está hoje ele não retém memória entre sessões. Se tentarmos rodar o invoke 2x (cria sessões diferentes) vemos que ele não se lembra do que foi dito na chamada anterior

```agentcore invoke "Oi, meu nome é Eduardo Fiuza"```

```agentcore invoke "Oi, qual é meu nome mesmo?"```

# Explicação inicial

- STM é a memória de curto prazo, é basicamente o texto completo da conversa recente.
- LTM é a memória de longo prazo, ela persiste conceitos-chave da conversa mas não o texto completo. 

Existem algumas estratégias para se utilizar a memória em um agente.
- SEMANTIC: Guarda pedaços factuais da conversa como nome, local, código. | Bom para recall pontual mas pode armazenar fato irrelevante e não captura o estilo nem sequência da conversa.
- SUMMARIZATION: Guarda o resumo comprimido da conversa na sessão. | A compressão pode ser boa para guardar o contexto da conversa mas pode acabar perdendo detalhes granulares dela. 
- USER_PREFERENCE: Guarda as preferencias do usuário como gosto, estilo, email preferido. | É bom para a personalização para o usuário mas assim como o SEMANTIC, também perde contexto do arco da conversa e também pode se engessar quanto a mundaça de preferências do usuário. 

# Adicionando Memória ao Agente

```agentcore add memory --name SharedMemory --strategies SEMANTIC,SUMMARIZATION --expiry 30```\
ou no Powershell:\
```agentcore add memory --name SharedMemory --strategies "SEMANTIC,SUMMARIZATION" --expiry 30``` 

- Harness default do agentcore utiliza as duas, vamos seguir o padrão
- a flag --expiry 30 serve para o tempo de vida da STM (30 dias)

```cat agentcore/agentcore.json``` Se quiser ver o arquivo de configuração do agente já com o objeto memória (Ou abra ele no diretório xD)

# Integrando a Memória ao Agente

O que criamos até agora foi o serviço "Memória", agora precisamos conectar ele ao agente

```
mkdir app\CustomerSupport\memory
New-Item app\CustomerSupport\memory\__init__.py -Force
New-Item app\CustomerSupport\memory\session.py -Force
```

Abra app/CustomerSupport/memory/session.py e cole o código:

**ATENÇÃO** A variável 'MEMORY_SHAREDMEMORY_ID' é injetada como variável de ambiente quando rodamos o 'agentcore deploy', o 'AWS_REGION' você pode colocar como variável de ambiente ou deixar hardcoded

```python
import os
from typing import Optional
from bedrock_agentcore.memory.integrations.strands.config import AgentCoreMemoryConfig, RetrievalConfig
from bedrock_agentcore.memory.integrations.strands.session_manager import AgentCoreMemorySessionManager

MEMORY_ID = os.getenv("MEMORY_SHAREDMEMORY_ID")
REGION = os.getenv("AWS_REGION", "sa-east-1")

def get_memory_session_manager(session_id: str, actor_id: str) -> Optional[AgentCoreMemorySessionManager]:
    if not MEMORY_ID:
        return None

    retrieval_config = {
        f"/users/{actor_id}/facts": RetrievalConfig(top_k=3, relevance_score=0.3),
        f"/summaries/{actor_id}/{session_id}": RetrievalConfig(top_k=3, relevance_score=0.3)
    }

    return AgentCoreMemorySessionManager(
        AgentCoreMemoryConfig(
            memory_id=MEMORY_ID,
            session_id=session_id,
            actor_id=actor_id,
            retrieval_config=retrieval_config,
        ),
        REGION
    )
```

# Update main.py para utilizar a memória

```python
from strands import Agent, tool
from bedrock_agentcore.runtime import BedrockAgentCoreApp
from model.load import load_model
from mcp_client.client import get_streamable_http_mcp_client
from memory.session import get_memory_session_manager

app = BedrockAgentCoreApp()
log = app.logger

# Define a Streamable HTTP MCP Client
mcp_clients = [get_streamable_http_mcp_client()]

# Resumos dos livros (chaves = títulos em português)
RESUMOS_LIVROS = {
    "Cinquenta Tons de Cinza": (
        "Anastasia Steele, estudante de literatura, entrevista o jovem bilionário "
        "Christian Grey para o jornal da faculdade. Os dois se envolvem em um "
        "relacionamento intenso, marcado pelos segredos e pelo passado conturbado "
        "de Grey, que obriga Ana a decidir até onde está disposta a ir."
    ),
    "Harry Potter e a Pedra Filosofal": (
        "Harry, um órfão criado pelos tios, descobre aos 11 anos que é bruxo e vai "
        "estudar em Hogwarts. Lá faz amizade com Rony e Hermione e descobre que "
        "alguém pretende roubar a Pedra Filosofal para trazer de volta Voldemort, "
        "o bruxo que matou seus pais."
    ),
    "Código Limpo": (
        "Guia prático sobre como escrever código legível e fácil de manter. "
        "Aborda nomes significativos, funções pequenas, comentários, formatação, "
        "tratamento de erros, testes unitários e refatoração, com vários "
        "estudos de caso em Java."
    ),
    "Gash Bell": (
        "Kiyomaro Takamine, um estudante genial e solitário, passa a cuidar de "
        "Gash, um menino-demônio sem memória. Gash participa de uma batalha "
        "entre 100 demônios para decidir o próximo rei do mundo demoníaco, e cada "
        "um precisa de um parceiro humano que leia o livro de feitiços."
    ),
    "O Hobbit": (
        "Bilbo Bolseiro, um hobbit pacato, é recrutado pelo mago Gandalf e por "
        "treze anões para recuperar o tesouro tomado pelo dragão Smaug. Na jornada "
        "enfrenta trolls, orcs e aranhas gigantes, e encontra o Um Anel."
    ),
}
 
# Informações dos livros
# Número de páginas: edição original (varia conforme a edição/tradução).
INFOS_LIVROS = {
    "Cinquenta Tons de Cinza": {"id": 1, "numero_paginas": 514, "autor": "E. L. James"},
    "Harry Potter e a Pedra Filosofal": {"id": 2, "numero_paginas": 223, "autor": "J. K. Rowling"},
    "Código Limpo": {"id": 3, "numero_paginas": 464, "autor": "Robert C. Martin"},
    # Mangá em 33 volumes: não há um número único de páginas
    "Gash Bell": {"id": 4, "numero_paginas": None, "autor": "Makoto Raiku"},
    "O Hobbit": {"id": 5, "numero_paginas": 310, "autor": "J. R. R. Tolkien"},
}

@tool
def obter_resumo_livro(nome_livro: str) -> str:
    """Obtém o resumo de um livro a partir do seu título em português.
 
    Args:
        nome_livro: título do livro, por exemplo "O Hobbit".
    """
    # Busca sem diferenciar maiúsculas/minúsculas e ignorando espaços nas pontas
    chave = nome_livro.strip().lower()
    for titulo, resumo in RESUMOS_LIVROS.items():
        if titulo.lower() == chave:
            return resumo or "Não existe resumo para este livro ainda"
    return "Não existe resumo para este livro ainda"


@tool
def obter_informacoes_livro(nome_livro: str) -> str:
    """Obtém as informações (id, número de paginas e nome do autor) de um livro a partir do seu título em português.
 
    Args:
        nome_livro: título do livro, por exemplo "O Hobbit".
    """
    # Busca sem diferenciar maiúsculas/minúsculas e ignorando espaços nas pontas
    chave = nome_livro.strip().lower()
    for titulo, infos in INFOS_LIVROS.items():
        if titulo.lower() == chave:
            return infos or "Não existe resumo para este livro ainda"
    return "Não existe resumo para este livro ainda"

tools = [obter_resumo_livro, obter_informacoes_livro]

# Add MCP client to tools if available
for mcp_client in mcp_clients:
    if mcp_client:
        tools.append(mcp_client)


_agent = None

def get_or_create_agent(session_id: str, user_id: str):
    global _agent
    if _agent is None:
        _agent = Agent(
            model=load_model(),
            session_manager=get_memory_session_manager(session_id, user_id),
            system_prompt="""
                Você é uma bibliotecária útil.

                Seu papel é Dar informações precisas aos clientes sobre os livros da biblioteca. 
            
                Você tem acesso à tools que te ajudam a encontrar resumos dos livros para ajudar na indicação e também outra que possui informações básicas sobre estes livros.
            """,
            tools=tools,
            callback_handler=None
        )
    return _agent

@app.entrypoint
async def invoke(payload, context):
    log.info("Invoking Agent.....")

    session_id = context.session_id

    headers = context.request_headers or {}
    user_id = headers.get("x-amzn-bedrock-agentcore-runtime-custom-user-id")

    if not session_id or not user_id:
        raise ValueError("session_id and user_id are required. Pass session-id and --user-id when invoking")
    

    agent = get_or_create_agent(session_id, user_id)
    stream = agent.stream_async(payload.get("prompt"))
    async for event in stream:
        if "data" in event and isinstance(event["data"], str):
            yield event["data"]


if __name__ == "__main__":
    app.run()

```

# Alterar o agentcore/agentcore.json

Por padrão, o Runtime do AgentCore não encaminha headers customizados ao main.py (injection/leak). Como a memória precisa de um user-id/actor_id para isolar a sessão por usuário, autorizamos o header X-Amzn-Bedrock-AgentCore-Runtime-Custom-User-Id na requestHeaderAllowlist do agentcore.json, para que o valor enviado na invocação chegue ao main.py.

```
"runtimes": [
    {
        "name": "Bibliotecario",
        "build": "CodeZip",
        "entrypoint": "main.py",
        "codeLocation": "app/Bibliotecario/",
        "runtimeVersion": "PYTHON_3_14",
        "networkMode": "PUBLIC",
        "protocol": "HTTP",
        "requestHeaderAllowlist": [
            "X-Amzn-Bedrock-AgentCore-Runtime-Custom-User-Id"
        ]
    }
]
```

# Deploy

```agentcore deploy -y -v```

### Arquitetura até aqui

![Serviço de memória adicionado](imagens\lab2_architecture_diagram.png)

# Testando a memória

### Linux

```
SESSION_A=$(python3 -c 'import uuid; print(uuid.uuid4())')
agentcore invoke "My name is Sarah and I prefer email updates. I recently bought a Smart Watch." \
  --session-id $SESSION_A \
  -H "X-Amzn-Bedrock-AgentCore-Runtime-Custom-User-Id: Sarah" --stream
```

Espera 2 minutos

```
sleep 2m
SESSION_B=$(python3 -c 'import uuid; print(uuid.uuid4())')
agentcore invoke "Do you know anything about me?" \
  --session-id $SESSION_B \
  -H "X-Amzn-Bedrock-AgentCore-Runtime-Custom-User-Id: Sarah" --stream
```

Repare que foi uma nova sessão (SESSION_A e SESSION_B mas com a mesma usuária "Sarah")

### Windows

```
$SESSION_A = [guid]::NewGuid().ToString()
agentcore invoke "Meu nome é Eduardo e eu gosto de ler O Hobbit." `
  --session-id $SESSION_A `
  -H "X-Amzn-Bedrock-AgentCore-Runtime-Custom-User-Id: Sarah" --stream

```

Espera 2 minutos

```
Start-Sleep -Seconds 120
$SESSION_B = [guid]::NewGuid().ToString()
agentcore invoke "Você sabe algo sobre mim?" `
  --session-id $SESSION_B `
  -H "X-Amzn-Bedrock-AgentCore-Runtime-Custom-User-Id: Sarah" --stream

```

Repare que foi uma nova sessão (SESSION_A e SESSION_B mas com a mesma usuária "Sarah")